# Networks, Systems, and Biological Complexity Workflow

This notebook scaffold supports biological network construction, adjacency matrices, degree summaries, density, module summaries, diffusion, robustness simulation, food-web scaffolds, microbiome association scaffolds, and provenance documentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
edges = pd.read_csv(article_dir / 'data' / 'biological_network_edges.csv')
nodes = sorted(set(edges['source']).union(edges['target']))
node_index = {node: i for i, node in enumerate(nodes)}
adjacency = np.zeros((len(nodes), len(nodes)))

for _, row in edges.iterrows():
    i = node_index[row['source']]
    j = node_index[row['target']]
    adjacency[i, j] = row['weight']
    adjacency[j, i] = row['weight']

pd.DataFrame(adjacency, index=nodes, columns=nodes).round(3)

In [ ]:
degree = (adjacency > 0).sum(axis=1)
weighted_degree = adjacency.sum(axis=1)
summary = pd.DataFrame({'node': nodes, 'degree': degree, 'weighted_degree': weighted_degree})
summary.sort_values('degree', ascending=False).round(4)

In [ ]:
initial = pd.read_csv(article_dir / 'data' / 'diffusion_initial_state.csv')
state_map = dict(zip(initial['node'], initial['initial_state']))
state = np.array([state_map.get(node, 0.0) for node in nodes])

alpha = 0.08
decay = 0.04
for _ in range(25):
    state = state + alpha * adjacency @ state - decay * state
    state = np.maximum(state, 0)

pd.DataFrame({'node': nodes, 'final_state': state}).sort_values('final_state', ascending=False).round(5)